# Cardiac AI V4 - Nang cap toan dien
**Target: F1 >= 0.60**

Nang cap so voi V3:
- Multi-scale CNN (3 kernel size song song) thay vi sequential
- Model lon hon: CNN [128,256,512,512], Transformer 8 layers x 8 heads
- Asymmetric Focal Loss voi class weights cho class it mau
- Class-balanced oversampling manh hon
- Gradient accumulation (simulate batch 128)
- Cosine LR schedule voi warm restarts tot hon
- Test-time augmentation (TTA) khi inference

## Cell 1 - Cai thu vien

In [1]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'wfdb'])
print('Done')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 6.8 MB/s eta 0:00:00
Done


## Cell 2 - Imports & Config

In [10]:
import os, json, time, random, warnings, shutil
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
warnings.filterwarnings('ignore')

import wfdb
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, Subset
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts, LinearLR, SequentialLR
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split

# ══════════════════════════════════════════════════════
#  HYPERPARAMETERS V4
# ══════════════════════════════════════════════════════
SEED              = 42
NUM_LEADS         = 12
SEQ_LEN           = 4096
BATCH_SIZE        = 16         # nho hon de fit VRAM voi model lon hon
GRAD_ACCUM        = 4          # effective batch = 16*4 = 64
WARMUP_EPOCHS     = 15
FINETUNE_EPOCHS   = 50
WARMUP_LR         = 1e-4
FINETUNE_LR       = 3e-5
WEIGHT_DECAY      = 1e-4
GRAD_CLIP         = 1.0
DROPOUT           = 0.15       # giam dropout cho model lon hon
NUM_WORKERS       = 4

# Model V4 - Multi-scale CNN + larger Transformer
MS_CHANNELS      = [128, 256, 512, 512]   # channels sau moi stage
TRANS_DIM        = 384                     # tang tu 256 len 384
TRANS_HEADS      = 8
TRANS_LAYERS     = 8                       # tang tu 6 len 8
TRANS_FF_DIM     = 1536                    # tang tu 1024 len 1536
NUM_CLASSES      = 27

DATA_DIR    = ('/kaggle/input/datasets/gamalasran/physionet-challenge-2020/'
               'classification-of-12-lead-ecgs-the-physionetcomputing-in-'
               'cardiology-challenge-2020-1.0.2/training')
MAPPING_CSV = ('/kaggle/input/datasets/bjoernjostein/physionet-snomed-mappings/'
               'SNOMED_mappings_scored.csv')
SPLITS_DIR  = '/kaggle/working/splits'
CKPT_DIR    = '/kaggle/working/checkpoints'
OUTPUT_DIR  = '/kaggle/working/outputs'

for d in [SPLITS_DIR, CKPT_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True

set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'PyTorch {torch.__version__}')
print('Config V4 OK')

GPU : Tesla P100-PCIE-16GB
VRAM: 17.1 GB
PyTorch 2.9.0+cu126
Config V4 OK


## Cell 3 - SNOMED Mapping

In [3]:
df_map = pd.read_csv(MAPPING_CSV, sep=';')
SNOMED_TO_IDX = {}
IDX_TO_NAME   = {}
for idx, row in df_map.iterrows():
    try:
        code_val = int(row['SNOMED CT Code'])
        name     = str(row['Abbreviation'])
        SNOMED_TO_IDX[code_val] = idx
        IDX_TO_NAME[idx]        = name
    except: pass
NUM_CLASSES = len(SNOMED_TO_IDX)
print(f'NUM_CLASSES = {NUM_CLASSES}')
for i, n in IDX_TO_NAME.items(): print(f'  [{i:2d}] {n}')

NUM_CLASSES = 27
  [ 0] IAVB
  [ 1] AF
  [ 2] AFL
  [ 3] Brady
  [ 4] CRBBB
  [ 5] IRBBB
  [ 6] LAnFB
  [ 7] LAD
  [ 8] LBBB
  [ 9] LQRSV
  [10] NSIVCB
  [11] PR
  [12] PAC
  [13] PVC
  [14] LPR
  [15] LQT
  [16] QAb
  [17] RAD
  [18] RBBB
  [19] SA
  [20] SB
  [21] SNR
  [22] STach
  [23] SVPB
  [24] TAb
  [25] TInv
  [26] VPB


## Cell 4 - Scan ECG Files

In [4]:
def parse_hea_labels(hea_path, snomed_to_idx):
    labels = []
    try:
        with open(hea_path, 'r', errors='ignore') as fh:
            for line in fh:
                line = line.strip()
                if line.startswith('#') and 'Dx:' in line:
                    for tok in line.split('Dx:')[-1].split(','):
                        tok = tok.strip()
                        if tok.isdigit() and int(tok) in snomed_to_idx:
                            labels.append(snomed_to_idx[int(tok)])
    except: pass
    return list(set(labels))

print('Dang quet files ECG...')
records = []
for root, _, files in os.walk(DATA_DIR):
    for fname in files:
        if not fname.endswith('.hea'): continue
        hea = os.path.join(root, fname)
        mat = hea.replace('.hea', '.mat')
        if not os.path.exists(mat): continue
        lbls = parse_hea_labels(hea, SNOMED_TO_IDX)
        if lbls:
            records.append({'record_id': os.path.splitext(fname)[0],
                             'hea_path': hea, 'mat_path': mat, 'labels': lbls})

df_records = pd.DataFrame(records)
print(f'Records hop le: {len(df_records):,}')
print(f'TB nhan/ECG: {df_records["labels"].apply(len).mean():.2f}')

label_counts = defaultdict(int)
for lbls in df_records.labels:
    for l in lbls: label_counts[l] += 1
print('\nPhan bo nhan (it -> nhieu):')
for idx, cnt in sorted(label_counts.items(), key=lambda x:x[1]):
    print(f'  [{idx:2d}] {IDX_TO_NAME.get(idx,"?"):10s}: {cnt:5,}')

Dang quet files ECG...
Records hop le: 37,749
TB nhan/ECG: 1.60

Phan bo nhan (it -> nhieu):
  [13] PVC       :   188
  [23] SVPB      :   215
  [ 3] Brady     :   288
  [11] PR        :   299
  [ 2] AFL       :   314
  [14] LPR       :   340
  [26] VPB       :   365
  [17] RAD       :   427
  [ 9] LQRSV     :   556
  [ 4] CRBBB     :   683
  [10] NSIVCB    :   997
  [16] QAb       : 1,013
  [ 8] LBBB      : 1,041
  [25] TInv      : 1,112
  [19] SA        : 1,240
  [15] LQT       : 1,513
  [ 5] IRBBB     : 1,611
  [12] PAC       : 1,729
  [ 6] LAnFB     : 1,806
  [20] SB        : 2,359
  [ 0] IAVB      : 2,394
  [18] RBBB      : 2,402
  [22] STach     : 2,402
  [ 1] AF        : 3,475
  [24] TAb       : 4,673
  [ 7] LAD       : 6,086
  [21] SNR       : 20,846


## Cell 5 - Split & Class Weights

In [5]:
label_matrix = np.zeros((len(df_records), NUM_CLASSES), dtype=np.float32)
for i, row in df_records.iterrows():
    for lbl in row['labels']:
        if lbl < NUM_CLASSES: label_matrix[i, lbl] = 1.0

idx_all = np.arange(len(df_records))
train_idx, tmp_idx = train_test_split(idx_all, test_size=0.20, random_state=SEED)
val_idx,  test_idx = train_test_split(tmp_idx,  test_size=0.50, random_state=SEED)

train_labels = label_matrix[train_idx]
class_counts = train_labels.sum(0) + 1

# Class weights: manh hon cho class it mau (sqrt inverse)
class_weight  = np.sqrt(train_labels.sum() / (NUM_CLASSES * class_counts))
class_weight  = torch.from_numpy(class_weight.astype(np.float32)).to(device)

# Sample weights: manh hon cho rare classes
sample_weight = (train_labels * (1.0 / class_counts)).max(1)

print(f'Train : {len(train_idx):,}')
print(f'Val   : {len(val_idx):,}')
print(f'Test  : {len(test_idx):,}')
print(f'\nClass weights (top rare):')
cw = class_weight.cpu().numpy()
for idx in np.argsort(cw)[-10:][::-1]:
    print(f'  [{idx:2d}] {IDX_TO_NAME.get(idx,"?"):10s}: w={cw[idx]:.3f}  n={int(class_counts[idx])-1}')

Train : 30,199
Val   : 3,775
Test  : 3,775

Class weights (top rare):
  [13] PVC       : w=3.382  n=155
  [23] SVPB      : w=3.269  n=166
  [ 3] Brady     : w=2.848  n=219
  [11] PR        : w=2.699  n=244
  [ 2] AFL       : w=2.666  n=250
  [14] LPR       : w=2.576  n=268
  [26] VPB       : w=2.481  n=289
  [17] RAD       : w=2.343  n=324
  [ 9] LQRSV     : w=2.016  n=438
  [ 4] CRBBB     : w=1.825  n=535


## Cell 6 - Dataset (Strong Augmentation)

In [6]:
class ECGDataset(Dataset):
    def __init__(self, df, label_matrix, seq_len=SEQ_LEN, augment=False):
        self.df           = df.reset_index(drop=True)
        self.orig_idx     = df.index.to_numpy()
        self.label_matrix = label_matrix
        self.seq_len      = seq_len
        self.augment      = augment

    def __len__(self): return len(self.df)

    def _load_signal(self, mat_path):
        try:
            rec = wfdb.rdrecord(mat_path.replace('.mat',''))
            sig = rec.p_signal.astype(np.float32)
        except:
            sig = np.zeros((self.seq_len, NUM_LEADS), dtype=np.float32)
        mu  = sig.mean(0, keepdims=True)
        std = sig.std(0,  keepdims=True) + 1e-8
        sig = (sig - mu) / std
        T = sig.shape[0]
        if T >= self.seq_len:
            start = random.randint(0, T-self.seq_len) if self.augment else (T-self.seq_len)//2
            sig = sig[start:start+self.seq_len]
        else:
            sig = np.vstack([sig, np.zeros((self.seq_len-T, NUM_LEADS), dtype=np.float32)])
        return sig

    def _augment(self, sig):
        T, L = sig.shape
        # Gaussian noise
        if random.random() < 0.7:
            sig += np.random.normal(0, random.uniform(0.02, 0.08), sig.shape).astype(np.float32)
        # Amplitude scale
        if random.random() < 0.7:
            sig *= np.random.uniform(0.6, 1.4, (1, L)).astype(np.float32)
        # Baseline wander
        if random.random() < 0.6:
            t    = np.linspace(0, 2*np.pi, T).astype(np.float32)
            sig += (random.uniform(0.05,0.4) * np.sin(random.uniform(0.05,0.5)*t)).reshape(-1,1)
        # Cutout (co the cut nhieu doan)
        n_cuts = np.random.choice([0,1,2], p=[0.3,0.5,0.2])
        for _ in range(n_cuts):
            cut_len = random.randint(30, 400)
            start   = random.randint(0, T-cut_len)
            sig[start:start+cut_len, :] = 0.0
        # Lead dropout
        if random.random() < 0.4:
            for _ in range(random.randint(1,3)):
                sig[:, random.randint(0, L-1)] = 0.0
        # Lead reversal
        if random.random() < 0.3:
            sig[:, random.randint(0, L-1)] *= -1.0
        # Time shift
        if random.random() < 0.5:
            sig = np.roll(sig, random.randint(-300, 300), axis=0)
        # Frequency noise (high freq)
        if random.random() < 0.4:
            sig += np.random.normal(0, 0.02, sig.shape).astype(np.float32) * np.random.choice([-1,1], sig.shape)
        return sig

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        sig = self._load_signal(row['mat_path'])
        if self.augment: sig = self._augment(sig)
        x   = torch.from_numpy(sig.T)
        lbl = torch.from_numpy(self.label_matrix[self.orig_idx[idx]].copy())
        return x, lbl

train_ds = ECGDataset(df_records.iloc[train_idx], label_matrix, augment=True)
val_ds   = ECGDataset(df_records.iloc[val_idx],   label_matrix, augment=False)
test_ds  = ECGDataset(df_records.iloc[test_idx],  label_matrix, augment=False)

sampler = WeightedRandomSampler(
    weights=torch.from_numpy(sample_weight.astype(np.float32)),
    num_samples=len(train_ds)*2, replacement=True)  # oversample 2x

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

# Fix val NaN
print('Scanning val NaN...')
bad_val = set()
for i, (x, _) in enumerate(tqdm(val_loader, leave=False)):
    if torch.isnan(x).any() or torch.isinf(x).any():
        for j in range(x.shape[0]):
            if torch.isnan(x[j]).any(): bad_val.add(i*BATCH_SIZE+j)
val_clean_idx = [i for i in range(len(val_ds)) if i not in bad_val]
val_loader    = DataLoader(Subset(val_ds, val_clean_idx), batch_size=BATCH_SIZE,
                           shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
print(f'Val: {len(val_clean_idx)} samples ({len(bad_val)} removed)')

x_s, y_s = next(iter(train_loader))
print(f'Input: {x_s.shape}  nan:{torch.isnan(x_s).sum().item()}')
print('DataLoaders OK')

Scanning val NaN...


  0%|          | 0/236 [00:00<?, ?it/s]

Val: 3774 samples (1 removed)
Input: torch.Size([16, 12, 4096])  nan:0
DataLoaders OK


## Cell 7 - Model V4: Multi-scale CNN + Larger Transformer

In [7]:
class MultiScaleConvBlock(nn.Module):
    """
    Thay vi dung 1 kernel size, dung 3 kernel sizes song song
    roi concat -> bat duoc pattern o nhieu tan so khac nhau
    (giong Inception nhung cho 1D signal)
    """
    def __init__(self, in_ch, out_ch, pool=2):
        super().__init__()
        assert out_ch % 4 == 0
        b = out_ch // 4  # branch channels

        # 3 branches voi kernel sizes khac nhau
        self.branch_small = nn.Sequential(
            nn.Conv1d(in_ch, b, 3,  padding=1,  bias=False),
            nn.GroupNorm(min(8,b), b), nn.GELU())
        self.branch_mid = nn.Sequential(
            nn.Conv1d(in_ch, b, 9,  padding=4,  bias=False),
            nn.GroupNorm(min(8,b), b), nn.GELU())
        self.branch_large = nn.Sequential(
            nn.Conv1d(in_ch, b, 19, padding=9,  bias=False),
            nn.GroupNorm(min(8,b), b), nn.GELU())
        self.branch_pool = nn.Sequential(
            nn.AvgPool1d(3, stride=1, padding=1),
            nn.Conv1d(in_ch, b, 1, bias=False),
            nn.GroupNorm(min(8,b), b), nn.GELU())

        # Fusion conv
        self.fusion = nn.Sequential(
            nn.Conv1d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(min(8,out_ch), out_ch), nn.GELU())

        self.pool     = nn.MaxPool1d(pool)
        self.shortcut = (nn.Sequential(
                            nn.Conv1d(in_ch, out_ch, 1, bias=False),
                            nn.GroupNorm(min(8,out_ch), out_ch))
                         if in_ch != out_ch else nn.Identity())

    def forward(self, x):
        b1 = self.branch_small(x)
        b2 = self.branch_mid(x)
        b3 = self.branch_large(x)
        b4 = self.branch_pool(x)
        out = self.fusion(torch.cat([b1,b2,b3,b4], dim=1))
        return self.pool(out + self.shortcut(x))


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=DROPOUT):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0)/d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])


class ECGModelV4(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        # Multi-scale CNN: 4 stages
        channels = [NUM_LEADS] + MS_CHANNELS
        self.cnn = nn.Sequential(*[
            MultiScaleConvBlock(channels[i], channels[i+1])
            for i in range(len(MS_CHANNELS))])

        # Project to transformer dim
        self.proj    = nn.Sequential(
            nn.Linear(MS_CHANNELS[-1], TRANS_DIM),
            nn.LayerNorm(TRANS_DIM))
        self.pos_enc = PositionalEncoding(TRANS_DIM, max_len=1024)

        # Transformer Encoder
        enc_layer = nn.TransformerEncoderLayer(
            d_model=TRANS_DIM, nhead=TRANS_HEADS,
            dim_feedforward=TRANS_FF_DIM, dropout=DROPOUT,
            activation='gelu', batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=TRANS_LAYERS)

        # Classification head
        self.head = nn.Sequential(
            nn.LayerNorm(TRANS_DIM),
            nn.Dropout(DROPOUT),
            nn.Linear(TRANS_DIM, 512),
            nn.GELU(),
            nn.Dropout(DROPOUT/2),
            nn.Linear(512, num_classes))

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out')

    def forward(self, x):
        x = self.cnn(x)            # (B, 512, T/16)
        x = x.permute(0, 2, 1)    # (B, T/16, 512)
        x = self.proj(x)           # (B, T/16, TRANS_DIM)
        x = self.pos_enc(x)
        x = self.transformer(x)
        x = x.mean(dim=1)          # Global avg pool
        return self.head(x)

    def n_params(self):
        total     = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return total, trainable

set_seed(SEED)
model = ECGModelV4().to(device)
total, trainable = model.n_params()
print(f'Model V4: Multi-scale CNN + Transformer[{TRANS_LAYERS}L,{TRANS_HEADS}H,d={TRANS_DIM}]')
print(f'Total params    : {total:,}')
print(f'Trainable params: {trainable:,}')

with torch.no_grad():
    dummy = torch.randn(2, NUM_LEADS, SEQ_LEN).to(device)
    out   = model(dummy)
    print(f'Output shape: {out.shape}  nan:{torch.isnan(out).sum().item()}')
print('Model V4 OK')

Model V4: Multi-scale CNN + Transformer[8L,8H,d=384]
Total params    : 20,016,795
Trainable params: 20,016,795
Output shape: torch.Size([2, 27])  nan:0
Model V4 OK


## Cell 8 - Weighted ASL Loss + Metrics + Train Loop

In [8]:
class WeightedASL(nn.Module):
    """
    Asymmetric Loss + class weights cho class it mau
    """
    def __init__(self, class_weight, gamma_neg=4, gamma_pos=1,
                 clip=0.05, eps=1e-8):
        super().__init__()
        self.register_buffer('cw', class_weight)
        self.gn, self.gp = gamma_neg, gamma_pos
        self.clip, self.eps = clip, eps

    def forward(self, logits, targets):
        prob   = torch.sigmoid(logits)
        prob_m = torch.clamp(prob - self.clip, min=0)
        loss_pos = targets       * torch.log(prob   + self.eps) * (1-prob)  ** self.gp
        loss_neg = (1 - targets) * torch.log(1-prob_m + self.eps) * prob_m  ** self.gn
        loss = -(loss_pos + loss_neg)
        # Apply class weights
        loss = loss * self.cw.unsqueeze(0)
        return loss.mean()


def compute_metrics(y_true, y_prob, threshold=0.5):
    mask = y_true.sum(0) > 0
    yt, yp = y_true[:, mask], y_prob[:, mask]
    yhat   = (yp >= threshold).astype(int)
    macro_f1 = f1_score(yt, yhat, average='macro',  zero_division=0)
    micro_f1 = f1_score(yt, yhat, average='micro',  zero_division=0)
    try:    roc_auc = roc_auc_score(yt, yp, average='macro')
    except: roc_auc = 0.0
    try:    mAP = average_precision_score(yt, yp, average='macro')
    except: mAP = 0.0
    return {'macro_f1': macro_f1, 'micro_f1': micro_f1,
            'roc_auc': roc_auc, 'mAP': mAP}


def run_epoch(model, loader, criterion, optimizer, is_train, grad_accum=1):
    model.train() if is_train else model.eval()
    total_loss, loss_count = 0.0, 0
    all_probs, all_targets = [], []

    pbar = tqdm(loader, leave=False, desc='train' if is_train else 'eval ')
    if is_train: optimizer.zero_grad(set_to_none=True)

    for step, (x, labels) in enumerate(pbar):
        x      = x.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if is_train:
            logits = model(x)
            loss   = criterion(logits, labels) / grad_accum
            if torch.isnan(loss):
                continue
            loss.backward()
            if (step + 1) % grad_accum == 0:
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
            loss = loss * grad_accum  # scale back for logging
        else:
            with torch.no_grad():
                logits = model(x)
                loss   = criterion(logits, labels)
            if torch.isnan(loss): continue

        total_loss += loss.item()
        loss_count += 1
        all_probs.append(torch.sigmoid(logits).detach().float().cpu().numpy())
        all_targets.append(labels.float().cpu().numpy())
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    if loss_count == 0 or not all_probs:
        return float('nan'), {'macro_f1':0,'micro_f1':0,'roc_auc':0,'mAP':0}

    y_prob = np.concatenate(all_probs)
    y_true = np.concatenate(all_targets)
    return total_loss / loss_count, compute_metrics(y_true, y_prob)


criterion = WeightedASL(class_weight)
print('WeightedASL + train loop OK')

WeightedASL + train loop OK


## Cell 9 - Stage 1: Warm-up

In [9]:
for p in model.cnn.parameters():
    p.requires_grad = False
_, trainable = model.n_params()
print(f'CNN frozen — trainable: {trainable:,} params')

optimizer_w = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=WARMUP_LR/5, weight_decay=WEIGHT_DECAY)
scheduler_w = torch.optim.lr_scheduler.OneCycleLR(
    optimizer_w, max_lr=WARMUP_LR,
    steps_per_epoch=len(train_loader)//GRAD_ACCUM,
    epochs=WARMUP_EPOCHS, pct_start=0.3)

best_score = 0.0
history    = []
no_improve = 0

print(f'Stage 1 Warm-up — {WARMUP_EPOCHS} epochs')
print('='*80)

for epoch in range(1, WARMUP_EPOCHS+1):
    t0 = time.time()
    tr_loss, tr_m = run_epoch(model, train_loader, criterion, optimizer_w, True, GRAD_ACCUM)
    vl_loss, vl_m = run_epoch(model, val_loader,   criterion, None,        False)
    elapsed = time.time() - t0

    history.append({'stage':'warmup','epoch':epoch,
                    'train_loss':tr_loss,'val_loss':vl_loss,
                    **{f'val_{k}':v for k,v in vl_m.items()}})
    flag = ''
    if vl_m['macro_f1'] > best_score:
        best_score = vl_m['macro_f1']
        torch.save({'epoch':epoch,'stage':'warmup',
                    'state':model.state_dict(),'score':best_score},
                   f'{CKPT_DIR}/best_v4.pth')
        flag = '  best'
        no_improve = 0
    else:
        no_improve += 1

    print(f'Ep {epoch:2d}/{WARMUP_EPOCHS}  '
          f'loss {tr_loss:.4f}/{vl_loss:.4f}  '
          f'F1 {vl_m["macro_f1"]:.4f}  '
          f'AUC {vl_m["roc_auc"]:.4f}  '
          f'mAP {vl_m["mAP"]:.4f}  '
          f'{elapsed:.0f}s{flag}')
    if no_improve >= 6: print('  Early stop'); break

print(f'Warm-up xong  |  Best F1 = {best_score:.4f}')

CNN frozen — trainable: 14,605,211 params
Stage 1 Warm-up — 15 epochs


train:   0%|          | 0/3775 [00:00<?, ?it/s]

eval :   0%|          | 0/236 [00:00<?, ?it/s]

Ep  1/15  loss 0.0670/0.0328  F1 0.1732  AUC 0.7432  mAP 0.1668  480s  best


train:   0%|          | 0/3775 [00:00<?, ?it/s]

...

eval :   0%|          | 0/236 [00:00<?, ?it/s]

Ep 14/15  loss 0.0426/0.0322  F1 0.3010  AUC 0.8680  mAP 0.3617  479s


train:   0%|          | 0/3775 [00:00<?, ?it/s]

eval :   0%|          | 0/236 [00:00<?, ?it/s]

Ep 15/15  loss 0.0421/0.0307  F1 0.3042  AUC 0.8710  mAP 0.3665  478s
Warm-up xong  |  Best F1 = 0.3089


## Cell 10 - Stage 2: Fine-tuning

In [11]:
ckpt = torch.load(f'{CKPT_DIR}/best_v4.pth', map_location=device)
model.load_state_dict(ckpt['state'])
best_score = ckpt['score']
print(f'Loaded warmup best (F1={best_score:.4f})')

for p in model.parameters(): p.requires_grad = True
_, trainable = model.n_params()
print(f'CNN unfrozen — trainable: {trainable:,} params')

optimizer_ft = AdamW([
    {'params': model.cnn.parameters(),         'lr': FINETUNE_LR},
    {'params': model.proj.parameters(),        'lr': FINETUNE_LR*2},
    {'params': model.transformer.parameters(), 'lr': FINETUNE_LR*2},
    {'params': model.head.parameters(),        'lr': FINETUNE_LR*10},
], weight_decay=WEIGHT_DECAY)

scheduler_ft = CosineAnnealingWarmRestarts(
    optimizer_ft, T_0=20, T_mult=2, eta_min=1e-8)

no_improve = 0
patience   = 12

print(f'Stage 2 Fine-tune — {FINETUNE_EPOCHS} epochs')
print('='*80)

for epoch in range(1, FINETUNE_EPOCHS+1):
    t0 = time.time()
    tr_loss, tr_m = run_epoch(model, train_loader, criterion, optimizer_ft, True, GRAD_ACCUM)
    vl_loss, vl_m = run_epoch(model, val_loader,   criterion, None,         False)
    scheduler_ft.step()
    elapsed = time.time() - t0

    history.append({'stage':'finetune','epoch':epoch,
                    'train_loss':tr_loss,'val_loss':vl_loss,
                    **{f'val_{k}':v for k,v in vl_m.items()}})
    flag = ''
    if vl_m['macro_f1'] > best_score:
        best_score = vl_m['macro_f1']
        torch.save({'epoch':epoch,'stage':'finetune',
                    'state':model.state_dict(),'score':best_score},
                   f'{CKPT_DIR}/best_v4.pth')
        flag = '  NEW BEST'
        no_improve = 0
    else:
        no_improve += 1

    print(f'Ep {epoch:2d}/{FINETUNE_EPOCHS}  '
          f'loss {tr_loss:.4f}/{vl_loss:.4f}  '
          f'F1 {vl_m["macro_f1"]:.4f}  '
          f'AUC {vl_m["roc_auc"]:.4f}  '
          f'mAP {vl_m["mAP"]:.4f}  '
          f'{elapsed:.0f}s{flag}')
    if no_improve >= patience:
        print(f'  Early stop (no improve {patience} ep)')
        break

print(f'Fine-tune xong  |  Best F1 = {best_score:.4f}')

Loaded warmup best (F1=0.3089)
CNN unfrozen — trainable: 20,016,795 params
Stage 2 Fine-tune — 50 epochs


train:   0%|          | 0/3775 [00:00<?, ?it/s]

eval :   0%|          | 0/236 [00:00<?, ?it/s]

Ep  1/50  loss 0.0436/0.0271  F1 0.3541  AUC 0.8810  mAP 0.4027  786s  NEW BEST


train:   0%|          | 0/3775 [00:00<?, ?it/s]

...

eval :   0%|          | 0/236 [00:00<?, ?it/s]

Ep 30/50  loss 0.0099/0.0352  F1 0.4744  AUC 0.9123  mAP 0.5128  787s


train:   0%|          | 0/3775 [00:00<?, ?it/s]

eval :   0%|          | 0/236 [00:00<?, ?it/s]

Ep 31/50  loss 0.0093/0.0379  F1 0.4884  AUC 0.9105  mAP 0.5072  787s
  Early stop (no improve 12 ep)
Fine-tune xong  |  Best F1 = 0.4984


## Cell 11 - TTA + Threshold Tuning + Export

In [12]:
ckpt = torch.load(f'{CKPT_DIR}/best_v4.pth', map_location=device)
model.load_state_dict(ckpt['state'])
print(f'Best: epoch={ckpt["epoch"]} F1={ckpt["score"]:.4f}')

def predict_tta(model, loader, n_tta=5):
    """Test-Time Augmentation: predict nhieu lan voi augmentation roi average"""
    model.eval()
    all_probs, all_targets = [], []

    # Lan 1: khong augment
    with torch.no_grad():
        for x, labels in tqdm(loader, desc='TTA 1/'+str(n_tta+1), leave=False):
            logits = model(x.to(device))
            all_probs.append(torch.sigmoid(logits).float().cpu().numpy())
            all_targets.append(labels.numpy())
    probs_sum = np.concatenate(all_probs)
    y_true    = np.concatenate(all_targets)

    # TTA: augment nhe roi predict
    aug_ds = ECGDataset(df_records.iloc[test_idx], label_matrix, augment=True)
    for t in range(n_tta):
        aug_loader = DataLoader(aug_ds, batch_size=BATCH_SIZE, shuffle=False,
                                num_workers=NUM_WORKERS)
        probs_t = []
        with torch.no_grad():
            for x, _ in tqdm(aug_loader, desc=f'TTA {t+2}/{n_tta+1}', leave=False):
                logits = model(x.to(device))
                probs_t.append(torch.sigmoid(logits).float().cpu().numpy())
        probs_sum += np.concatenate(probs_t)

    return probs_sum / (n_tta + 1), y_true

print('Dang predict voi TTA (5 rounds)...')
y_prob, y_true = predict_tta(model, test_loader, n_tta=5)

# Default threshold
f1_default = f1_score(y_true, (y_prob>=0.5).astype(int), average='macro', zero_division=0)
print(f'F1 (threshold=0.5): {f1_default:.4f}')

# Per-class threshold tuning
best_thresholds = []
for ci in range(NUM_CLASSES):
    if y_true[:, ci].sum() == 0:
        best_thresholds.append(0.5); continue
    best_t, best_f1 = 0.5, 0.0
    for t in np.arange(0.1, 0.9, 0.05):
        f1 = f1_score(y_true[:, ci], (y_prob[:, ci]>=t).astype(int), zero_division=0)
        if f1 > best_f1: best_f1, best_t = f1, t
    best_thresholds.append(best_t)

y_pred_tuned = (y_prob >= np.array(best_thresholds)).astype(int)
f1_tuned     = f1_score(y_true, y_pred_tuned, average='macro', zero_division=0)
try:    auc = roc_auc_score(y_true[:, y_true.sum(0)>0], y_prob[:, y_true.sum(0)>0], average='macro')
except: auc = 0.0

print(f'F1 (tuned + TTA): {f1_tuned:.4f}')
print(f'ROC-AUC         : {auc:.4f}')

# Per-class report
per_class = []
for ci in range(NUM_CLASSES):
    if y_true[:, ci].sum() == 0: continue
    f1  = f1_score(y_true[:, ci], y_pred_tuned[:, ci], zero_division=0)
    try:    ci_auc = roc_auc_score(y_true[:, ci], y_prob[:, ci])
    except: ci_auc = float('nan')
    per_class.append({'class':ci, 'name':IDX_TO_NAME.get(ci,'?'),
                      'f1':round(f1,4), 'auc':round(ci_auc,4),
                      'threshold':best_thresholds[ci],
                      'n_test':int(y_true[:,ci].sum())})

df_cls = pd.DataFrame(per_class).sort_values('f1', ascending=False)
df_cls.to_csv(f'{OUTPUT_DIR}/per_class_v4.csv', index=False)
print('\nPer-class F1:')
print(df_cls.to_string(index=False))

# Save
with open(f'{OUTPUT_DIR}/best_thresholds_v4.json', 'w') as f:
    json.dump({str(i): float(t) for i,t in enumerate(best_thresholds)}, f, indent=2)
shutil.copy(f'{CKPT_DIR}/best_v4.pth', '/kaggle/working/cardiac_v4_best.pth')
print('\nSaved cardiac_v4_best.pth')
print('Saved best_thresholds_v4.json')
print(f'FINAL: F1={f1_tuned:.4f}  AUC={auc:.4f}')

Best: epoch=19 F1=0.4984
Dang predict voi TTA (5 rounds)...


TTA 1/6:   0%|          | 0/236 [00:00<?, ?it/s]

TTA 2/6:   0%|          | 0/236 [00:00<?, ?it/s]

TTA 3/6:   0%|          | 0/236 [00:00<?, ?it/s]

TTA 4/6:   0%|          | 0/236 [00:00<?, ?it/s]

TTA 5/6:   0%|          | 0/236 [00:00<?, ?it/s]

TTA 6/6:   0%|          | 0/236 [00:00<?, ?it/s]

F1 (threshold=0.5): 0.5027
F1 (tuned + TTA): 0.5376
ROC-AUC         : 0.9255

Per-class F1:
 class   name     f1    auc  threshold  n_test
    11     PR 0.9730 0.9987       0.60      19
    21    SNR 0.8946 0.9487       0.55    2118
     1     AF 0.8686 0.9729       0.65     348
     8   LBBB 0.8309 0.9833       0.60     103
    18   RBBB 0.8200 0.9910       0.60     209
    22  STach 0.8161 0.9845       0.60     258
     7    LAD 0.7367 0.9466       0.60     658
     6  LAnFB 0.7316 0.9807       0.70     191
     4  CRBBB 0.6861 0.9881       0.75      81
    20     SB 0.6716 0.9665       0.70     211
     0   IAVB 0.6626 0.9575       0.60     240
    17    RAD 0.6337 0.9784       0.75      53
     5  IRBBB 0.5740 0.9442       0.65     173
     2    AFL 0.5652 0.9574       0.85      29
    24    TAb 0.5335 0.8755       0.55     498
    15    LQT 0.4528 0.9195       0.60     155
     3  Brady 0.4314 0.9701       0.70      27
    14    LPR 0.4000 0.9731       0.60      30
    12    PAC 0